>Note:
In this file, we want to do an experiment by splitting the most important feature (Smoker)  
and adding a Polynomial feature to clarify the Non-Linear pattern, then comparing it with the previous model.

## Import Library

In [34]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## Load Dataset

In [35]:
train_data = pd.read_pickle('../data/preprocess/train_data.pkl')
test_data = pd.read_pickle('../data/preprocess/test_data.pkl')

In [36]:
train_data.head(2)

,age,gender,bmi,bloodpressure,diabetic,children,smoker,claim,age_group,bmi_category,bp_status,reg_northwest,reg_southeast,reg_southwest
1148,32.0,1,26.7,115,1,1,1,26109.33,2,2,0,1,0,0
807,51.0,0,25.7,83,1,0,0,11454.02,3,2,0,1,0,0


## Feature Engginering

In [37]:
def create_interaction(df):
    df['smoker_bmi'] = df['smoker'] * df['bmi']
    df['smoker_age'] = df['smoker'] * df['age']
    return df

In [38]:
def create_polynomials(df):
    df['bmi_squared'] = df['bmi'] ** 2
    df['age_squared'] = df['age'] ** 2
    return df

In [39]:
def bmi_bp_ratio(df):
    df['bmi_bp_ratio'] = df['bmi'] / (df['bloodpressure'] + 1)
    return df

In [40]:
def health_risk_score(df):
    df['health_risk_score'] = ((df['smoker'] * 3) + (df['bmi'] > 30).astype(int) * 2 + (df['bloodpressure'] >=140).astype(int) * 1)
    return df

In [41]:
def pipeline_feat_engg(df):
    df = create_interaction(df)
    df = bmi_bp_ratio(df)
    df = create_polynomials(df)
    df = health_risk_score(df)
    return df

In [42]:
# call func
train_data = pipeline_feat_engg(train_data)
test_data = pipeline_feat_engg(test_data)

In [43]:
train_data.head()

,age,gender,bmi,bloodpressure,diabetic,children,smoker,claim,age_group,bmi_category,bp_status,reg_northwest,reg_southeast,reg_southwest,smoker_bmi,smoker_age,bmi_bp_ratio,bmi_squared,age_squared,health_risk_score
1148,32.0,1,26.7,115,1,1,1,26109.33,2,2,0,1,0,0,26.7,32.0,0.230172,712.89,1024.0,3
807,51.0,0,25.7,83,1,0,0,11454.02,3,2,0,1,0,0,0.0,0.0,0.305952,660.49,2601.0,0
1287,32.0,0,39.0,96,1,0,1,42983.46,2,3,0,1,0,0,39.0,32.0,0.402062,1521.00,1024.0,5
590,38.0,0,23.4,96,1,3,0,8252.28,3,1,0,0,0,0,0.0,0.0,0.241237,547.56,1444.0,0
1188,37.0,1,28.3,88,0,3,1,32787.46,3,2,0,1,0,0,28.3,37.0,0.317978,800.89,1369.0,3


## Simple EDA for check the correlation each feat

In [44]:
# Check Linear Cor

def check_corr(df):
    pearson_corr = df.corr(method='pearson')['claim']
    spearman_corr = df.corr(method='spearman')['claim']
    comparison = pd.DataFrame({'Pearson (Linear)': pearson_corr, 'Spearman (Non-Linear)': spearman_corr})
    
    # sort by the largest absolute value, 
    # but why Spearman? because we want to know about the features that have an overall influence
    comparison['abs_spearman'] = comparison['Spearman (Non-Linear)'].abs()
    # sort by abs_spearman column then delete that column so it doesn't get in the way
    comparison = comparison.sort_values(by='abs_spearman', ascending=False).drop(columns=['abs_spearman'])
    return comparison.style.background_gradient(cmap='coolwarm', axis=0)

check_corr(train_data)

,Pearson (Linear),Spearman (Non-Linear)
claim,1.000000,1.000000
smoker_bmi,0.836468,0.665175
smoker,0.777491,0.656681
smoker_age,0.731557,0.650287
health_risk_score,0.722491,0.535126
bloodpressure,0.536728,0.371222
bp_status,0.438985,0.323355
children,0.083779,0.143343
bmi_bp_ratio,-0.098452,-0.132117
bmi_category,0.184486,0.103299


## Prepare Data

In [45]:
X_train = train_data.drop('claim', axis=1)
y_train = train_data['claim']

X_test = test_data.drop('claim', axis=1)
y_test = test_data['claim']

## Test with model

Now we will try an experiment with:
- A -> model using full features
- B -> model using top features

The experiment was conducted to determine whether feature selection is necessary and to compare the MAE between the two experiments.

In [49]:
top_features = ['smoker_bmi', 'smoker', 'smoker_age', 'health_risk_score', 'bloodpressure', 'bp_status']

scenario = {
    'Advanced + Full_feat': X_train.columns.to_list(),
    'Advanced + Top_feat': top_features
}

models = {
    'RandomForest': RandomForestRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42)
} 

results = [
    {'Scenario': 'Baseline (NB 03)', 'Model': 'XGBoost', 'MAE': 3952.31, 'RMSE': 5517.47, 'R2': 0.81},
    {'Scenario': 'Baseline (NB 03)', 'Model': 'RandomForest', 'MAE': 4029.61, 'RMSE': 5376.69, 'R2': 0.82}
]


for s_name, f_list in scenario.items():
    for name, model in models.items():
        model.fit(X_train[f_list], y_train)
        preds = model.predict(X_test[f_list])
        
        mae = mean_absolute_error(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        r2 = r2_score(y_test, preds)
        
        results.append({
            'Scenario': s_name,
            'Model': name,
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2
        })

# ascending=True -> because we want sort by MAE from lowest to hightest
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by='MAE', ascending=True).reset_index(drop=True)
df_results_styled = df_results.style.format({
    'MAE': '{:,.2f}',
    'RMSE': '{:,.2f}',
    'R2': '{:,.2f}'
}) 
display(df_results_styled)

,Scenario,Model,MAE,RMSE,R2
0,Advanced + Top_feat,RandomForest,"3,817.95","5,101.77",0.84
1,Advanced + Top_feat,XGBoost,"3,890.78","5,196.32",0.84
2,Baseline (NB 03),XGBoost,"3,952.31","5,517.47",0.81
3,Advanced + Full_feat,XGBoost,"3,952.31","5,517.47",0.81
4,Baseline (NB 03),RandomForest,"4,029.61","5,376.69",0.82
5,Advanced + Full_feat,RandomForest,"4,029.61","5,376.69",0.82


## Save the Data and Model

In [55]:
train_data[top_features].to_pickle('../data/preprocess/n_train_data.pkl')
test_data[top_features].to_pickle('../data/preprocess/n_test_data.pkl')

for name, model in models.items():
    model_name = name.lower().replace(' ', '_')
    filename = f'../model/{model_name}_v1.pkl'
    joblib.dump(model, filename)
    print(f'Model {name} saved to {filename}')

Model RandomForest saved to ../model/randomforest_v1.pkl
Model XGBoost saved to ../model/xgboost_v1.pkl
